
# Start Here: Exact and Trotterized Hamiltonian Evolution

This notebook introduces the central pipeline used throughout the course:

$$
H
\quad\longrightarrow\quad
U(t)=e^{-iHt}
\quad\longrightarrow\quad
\widetilde U(t)
\quad\longrightarrow\quad
\text{error and computational cost}.
$$

The symbols mean:

- \(H\): the Hamiltonian, which describes the dynamics of the system.
- \(U(t)\): the exact time-evolution operator.
- \(\widetilde U(t)\): an approximation to the exact evolution.
- Error: a measure of how different the approximation is from the exact result.

The main goal is not to memorize the code. It is to understand what mathematical object each line creates.



## 1. Make the repository code available to Python

The functions used in this notebook live inside the repository's `src` folder.  
The next cell adds the repository root to Python's search path.

This is a Python setup step, not a quantum-mechanics calculation.


In [16]:

import sys
from pathlib import Path

current_folder = Path.cwd().resolve()

# This works whether the notebook is launched from the repository root
# or from the notebooks/ directory.
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Repository root:", project_root)


Repository root: D:\github\Fault-Tolerant-Hamiltonian-Simulation



## 2. Import the numerical tools

We import:

- NumPy for numerical arrays;
- a function that builds the transverse-field Ising Hamiltonian;
- a function that creates computational-basis states;
- functions for exact and approximate time evolution;
- functions for measuring approximation error.

A system of \(n\) qubits has a Hilbert-space dimension

$$
d=2^n.
$$

Therefore:

- a state is a vector with \(2^n\) complex entries;
- an operator is a \(2^n\times 2^n\) complex matrix.

This exponential growth is the reason exact dense-matrix simulation eventually becomes impossible on a classical computer.


In [17]:

import numpy as np

from src.ft_ham_sim.hamiltonians import tfim_components
from src.ft_ham_sim.operators import basis_state
from src.ft_ham_sim.trotter import exact_unitary, trotter_unitary
from src.ft_ham_sim.metrics import operator_error, state_fidelity



## 3. Choose the physical model and simulation parameters

We use a one-dimensional transverse-field Ising model:

$$
H = H_{ZZ}+H_X.
$$

For an open chain of \(n\) qubits,

$$
H_{ZZ}
=
J\sum_{j=0}^{n-2} Z_jZ_{j+1},
$$

and

$$
H_X
=
h\sum_{j=0}^{n-1}X_j.
$$

Here:

- \(Z_jZ_{j+1}\) couples neighbouring qubits;
- \(X_j\) acts on one qubit and produces transitions between computational-basis states;
- \(J\) is the coupling strength;
- \(h\) is the transverse-field strength.

For the values used below,

$$
n=3,\qquad J=1,\qquad h=0.8.
$$

Thus,

$$
H_{ZZ}
=
Z_0Z_1+Z_1Z_2,
$$

and

$$
H_X
=
0.8\left(X_0+X_1+X_2\right).
$$

We also choose:

$$
t=1,
\qquad
r=4,
$$

where \(t\) is the total simulated time and \(r\) is the number of Trotter steps.


In [18]:

n = 3
t = 1.0
r = 4

h_zz, h_x = tfim_components(
    n,
    coupling=1.0,
    field=0.8,
)

print("Hilbert-space dimension:", 2**n)
print("Shape of H_ZZ:", h_zz.shape)
print("Shape of H_X:", h_x.shape)


Hilbert-space dimension: 8
Shape of H_ZZ: (8, 8)
Shape of H_X: (8, 8)



## 4. Exact time evolution

For a time-independent Hamiltonian, the exact time-evolution operator is

$$
U(t)=e^{-iHt},
$$

with

$$
H=H_{ZZ}+H_X.
$$

The evolved state is

$$
|\psi(t)\rangle
=
U(t)|\psi(0)\rangle.
$$

The function `exact_unitary` computes the matrix exponential directly.

This is practical only for small systems because \(U(t)\) is a dense matrix of size

$$
2^n\times 2^n.
$$



## 5. Why Trotterization is needed

If two operators commute,

$$
[A,B]=AB-BA=0,
$$

then

$$
e^{A+B}=e^Ae^B.
$$

However, the two parts of the Ising Hamiltonian generally do not commute:

$$
[H_{ZZ},H_X]\neq 0.
$$

Therefore,

$$
e^{-i(H_{ZZ}+H_X)t}
\neq
e^{-iH_{ZZ}t}e^{-iH_Xt}.
$$

Trotterization handles this by dividing the total time into shorter intervals,

$$
\Delta t=\frac{t}{r}.
$$

For a sufficiently small \(\Delta t\), the separated evolution is a useful approximation.



### First-order Lie–Trotter formula

The first-order approximation is

$$
U_1(t)
=
\left[
e^{-iH_{ZZ}\Delta t}
e^{-iH_X\Delta t}
\right]^r,
$$

where

$$
\Delta t=\frac{t}{r}.
$$

Its global error typically scales as

$$
\mathcal{O}\!\left(\frac{t^2}{r}\right).
$$

Increasing \(r\) usually reduces the approximation error, but it also increases the number of operations.



### Symmetric second-order formula

The second-order approximation uses a symmetric sequence:

$$
U_2(t)
=
\left[
e^{-iH_{ZZ}\Delta t/2}
e^{-iH_X\Delta t}
e^{-iH_{ZZ}\Delta t/2}
\right]^r.
$$

Its global error typically scales as

$$
\mathcal{O}\!\left(\frac{t^3}{r^2}\right).
$$

For sufficiently small time steps, second order should therefore converge faster than first order.


In [19]:

H = h_zz + h_x

# Exact evolution
U = exact_unitary(H, t)

# First-order product formula
U1 = trotter_unitary(
    (h_zz, h_x),
    t,
    r,
    order=1,
)

# Symmetric second-order product formula
U2 = trotter_unitary(
    (h_zz, h_x),
    t,
    r,
    order=2,
)

error_first = operator_error(U, U1)
error_second = operator_error(U, U2)

print("First-order operator error: ", error_first)
print("Second-order operator error:", error_second)


First-order operator error:  1.7090662923484206
Second-order operator error: 1.7106458331257604



## 6. Operator error

The notebook uses the spectral norm

$$
\epsilon_{\mathrm{op}}
=
\left\|
U-\widetilde U
\right\|_2.
$$

Equivalently,

$$
\left\|
U-\widetilde U
\right\|_2
=
\max_{\|\phi\|=1}
\left\|
\left(U-\widetilde U\right)|\phi\rangle
\right\|.
$$

This means that operator error measures the largest possible discrepancy over all normalized input states.

It is therefore a global, worst-case measure of the quality of the approximate evolution.



## 7. Choose an initial state

The expression

```python
basis_state("0" * n)
```

creates

$$
|\psi_0\rangle
=
|00\cdots 0\rangle.
$$

For \(n=3\),

$$
|\psi_0\rangle=|000\rangle.
$$

In vector form,

$$
|000\rangle
=
\begin{pmatrix}
1\\
0\\
0\\
0\\
0\\
0\\
0\\
0
\end{pmatrix}.
$$

We now evolve this same initial state with the exact and approximate operators:

$$
|\psi\rangle=U|\psi_0\rangle,
$$

$$
|\psi_1\rangle=U_1|\psi_0\rangle,
$$

$$
|\psi_2\rangle=U_2|\psi_0\rangle.
$$


In [20]:

psi0 = basis_state("0" * n)

psi = U @ psi0
psi1 = U1 @ psi0
psi2 = U2 @ psi0

infidelity_first = 1 - state_fidelity(psi, psi1)
infidelity_second = 1 - state_fidelity(psi, psi2)

print("First-order state infidelity: ", infidelity_first)
print("Second-order state infidelity:", infidelity_second)


First-order state infidelity:  0.6519009883622962
Second-order state infidelity: 0.6525996770785459



## 8. State fidelity and infidelity

For normalized pure states, the fidelity is

$$
F\left(\psi,\widetilde\psi\right)
=
\left|
\langle\psi|\widetilde\psi\rangle
\right|^2.
$$

Its range is

$$
0\leq F\leq 1.
$$

A fidelity of one means that the two physical states agree.  
The notebook reports the infidelity,

$$
1-F.
$$

Therefore:

- \(1-F=0\): perfect agreement;
- a small positive value: a good approximation;
- a larger value: a worse approximation.

Fidelity is insensitive to a global phase. Thus,

$$
|\widetilde\psi\rangle
=
e^{i\alpha}|\psi\rangle
$$

still gives

$$
F=1.
$$



## 9. Why operator error and state infidelity are different

Operator error asks:

> How different are the two evolution operators in the worst possible case?

State infidelity asks:

> How differently do they evolve this particular initial state?

Thus, it is possible to have a noticeable operator error while obtaining a very accurate result for one special state.

Mathematically,

$$
\left\|U-\widetilde U\right\|_2
$$

depends on the complete operators, whereas

$$
1-
\left|
\langle\psi_0|
U^\dagger\widetilde U
|\psi_0\rangle
\right|^2
$$

depends on the chosen state \( |\psi_0\rangle \).



## 10. Accuracy versus computational cost

Increasing the number of Trotter steps gives a shorter step size:

$$
\Delta t=\frac{t}{r}.
$$

This generally improves accuracy, but every additional step repeats the sequence of exponentials.

Schematically,

$$
\text{cost}\propto r.
$$

The basic resource-estimation trade-off is therefore

$$
\boxed{
\text{more Trotter steps}
\;\Longrightarrow\;
\text{smaller simulation error}
\;\text{but more quantum operations}
}.
$$


In [21]:

print(
    f"{'r':>4} "
    f"{'first-order error':>22} "
    f"{'second-order error':>23} "
    f"{'first-order infidelity':>27} "
    f"{'second-order infidelity':>28}"
)

for steps in [1, 2, 4, 8, 16, 32]:
    approx_1 = trotter_unitary(
        (h_zz, h_x),
        t,
        steps,
        order=1,
    )
    approx_2 = trotter_unitary(
        (h_zz, h_x),
        t,
        steps,
        order=2,
    )

    state_1 = approx_1 @ psi0
    state_2 = approx_2 @ psi0

    print(
        f"{steps:4d} "
        f"{operator_error(U, approx_1):22.8e} "
        f"{operator_error(U, approx_2):23.8e} "
        f"{1-state_fidelity(psi, state_1):27.8e} "
        f"{1-state_fidelity(psi, state_2):28.8e}"
    )


   r      first-order error      second-order error      first-order infidelity      second-order infidelity
   1         1.34637159e+00          8.54035731e-01              6.39790157e-01               4.53059460e-01
   2         5.77912043e-01          1.63331292e-01              9.45282496e-02               1.57756895e-02
   4         1.70906629e+00          1.71064583e+00              6.51900988e-01               6.52599677e-01
   8         1.64993331e+00          1.65004358e+00              6.17884645e-01               6.17809189e-01
  16         6.77611458e-02          2.40103073e-03              1.40224283e-03               3.16737532e-06
  32         3.38146849e-02          5.99838756e-04              3.52927919e-04               1.97529578e-07



## 11. Questions to answer after running the notebook

### Question 1

Why are operator error and state infidelity different?

**Hint:** one is a worst-case comparison of complete operators; the other concerns only the state \( |000\rangle \).

### Question 2

What happens when \(r\) is doubled?

Examine both the numerical error and the number of repeated product-formula steps.

### Question 3

Which computation becomes impossible first as \(n\) increases?

Remember that the state-vector length is

$$
2^n,
$$

while the number of entries in a dense operator is

$$
\left(2^n\right)^2=4^n.
$$

This is why storing exact dense evolution operators becomes impossible even earlier than storing state vectors.



## 12. What to remember from this notebook

The complete conceptual chain is

$$
H
\quad\longrightarrow\quad
U(t)=e^{-iHt}
\quad\longrightarrow\quad
|\psi(t)\rangle=U(t)|\psi(0)\rangle.
$$

When exact evolution is too expensive, we replace it by a product formula:

$$
\widetilde U(t)\approx U(t).
$$

We then quantify the approximation using either:

$$
\left\|U-\widetilde U\right\|_2,
$$

or

$$
1-
\left|
\langle\psi|\widetilde\psi\rangle
\right|^2.
$$

Finally, we compare the accuracy improvement with the additional computational cost.

That accuracy–cost trade-off is the first bridge from Hamiltonian simulation to quantum resource estimation.
